In [10]:
import os
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [12]:
raw_financial_data = (
    "ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки."
    "Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові."
    "ГрінЕнерджі Солюшнс за 2024 рік: виручка 890 млн дол (+35 відсотків), чистий прибуток 156 млн дол, маржа EBITDA 28 відсотків."
    "Працівників 5100 осіб. ROE становить 19.7 відсотка. Співвідношення борг до EBITDA дорівнює 0.9. Лідер у секторі відновлюваної енергії."
    "РітейлМакс Груп за 2024 рік: виручка 1200 млн дол (+8 відсотків), чистий прибуток 48 млн дол, маржа EBITDA 12 відсотків."
    "Працівників 8500 осіб. ROE становить 9.3 відсотка. Співвідношення борг до EBITDA дорівнює 3.2. Мережа з 340 магазинів в Україні."
    "ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсотки), чистий збиток мінус 15 млн дол, маржа EBITDA мінус 8 відсотків."
    "Працівників 890 осіб. Від'ємний показник ROE. Стартап який активно інвестує у масштабування бізнесу."
)

In [13]:
cleaned_data = re.sub(r'\.(?=[А-ЯІЄЇ])', '.\n\n', raw_financial_data)

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n"],
    chunk_size=300,
    chunk_overlap=0
)

docs = text_splitter.create_documents([cleaned_data])
print(f"Успішно створено чанків: {len(docs)}")

Успішно створено чанків: 4


In [14]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", base_url="https://openrouter.ai/api/v1")

vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="finansovi_zvity_kompaniy"
)

In [15]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

In [16]:
prompt_template = """Ти є професійним фінансовим аналітиком. Обов'язково використовуйте ТІЛЬКИ наданий контекст для відповіді на питання.
Якщо в наданому контексті немає інформації про компанію або запитуваний показник, чітко відповідайте: "На основі наданого контексту, інформація про ... відсутня."
Не вигадуйте факти та цифри. Відповідь має бути лаконічною та точною.

Контекст:
{context}

Питання: {question}
Відповідь:"""

prompt = ChatPromptTemplate.from_template(prompt_template)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, base_url="https://openrouter.ai/api/v1")

def format_docs(documents):
    return "\n\n".join(doc.page_content for doc in documents)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [17]:
test_questions = [
    "Яка виручка у компанії ТехНова Інк за 2024 рік?",
    "Скільки працівників у ГрінЕнерджі Солюшнс?",
    "Який показник ROE у РітейлМакс Груп?",
    "Де знаходиться штаб-квартира ТехНова Інк?",
    "Яка маржа EBITDA у ФінТех Дайнамікс?",
    "Який чистий прибуток у ЕкоПакування Солюшнс?"
]

print("=" * 70)
for i, question in enumerate(test_questions, 1):
    answer = rag_chain.invoke(question)
    print(f"Питання {i}: {question}")
    print(f"Відповідь: {answer}")
    print("-" * 70)

Питання 1: Яка виручка у компанії ТехНова Інк за 2024 рік?
Відповідь: Виручка у компанії ТехНова Інк за 2024 рік становить 450 млн дол.
----------------------------------------------------------------------
Питання 2: Скільки працівників у ГрінЕнерджі Солюшнс?
Відповідь: У ГрінЕнерджі Солюшнс 5100 працівників.
----------------------------------------------------------------------
Питання 3: Який показник ROE у РітейлМакс Груп?
Відповідь: ROE у РітейлМакс Груп становить 9.3 відсотка.
----------------------------------------------------------------------
Питання 4: Де знаходиться штаб-квартира ТехНова Інк?
Відповідь: Штаб-квартира ТехНова Інк знаходиться у Львові.
----------------------------------------------------------------------
Питання 5: Яка маржа EBITDA у ФінТех Дайнамікс?
Відповідь: Маржа EBITDA у ФінТех Дайнамікс становить мінус 8 відсотків.
----------------------------------------------------------------------
Питання 6: Який чистий прибуток у ЕкоПакування Солюшнс?
Відповідь: 